# M2 · Select body_shell or slab (T-005)

Set `SOURCE` below, then run all cells. Both routes call `roof.design_roof`.
The body route ends at design with an inner `b_min` keep. This notebook does not
apply slab water, openings, pores, fabrication or weathering to the body.
For the complete slab workflow, open [walkthrough.ipynb](walkthrough.ipynb).
All coordinates and thicknesses are mm; the preview uses nominal voxel=100 mm
and prints the actual source-grid spacing. T and b_min remain provisional.


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import grid_roof as roof
from grid_roof.checks import body_keep

SOURCE = "body_shell"                 # "body_shell" or "slab"
brief = roof.Brief(body_thickness=300, b_min=100, voxel=100)


In [ ]:
if SOURCE == "body_shell":
    mesh = roof.body.load_body("inputs/body_a.obj")  # TEST BOX in mm; no automatic scaling
    bbox = roof.body.mesh_bounds(mesh)
    mesh.translate([-(bbox[i] + bbox[i + 3]) / 2 for i in range(3)])
    result = roof.design_roof(brief, body_mesh=mesh)
    bounds = result.bounds
    print("TEST BOX; mesh bbox (mm):", bounds, "; nominal voxel (mm):", brief.voxel)
elif SOURCE == "slab":
    drawing = roof.Drawing.load("drawing.json")
    result = roof.design_roof(brief, drawing)
    bounds = brief.bounds
else:
    raise ValueError('SOURCE must be "body_shell" or "slab"')
design, keep = result.design, result.keep


In [ ]:
# A world-space XZ section through bbox mid-Y, including negative coordinates.
pad = 2 * brief.voxel
y = (bounds[1] + bounds[4]) / 2
xs = np.arange(bounds[0] - pad, bounds[3] + pad, brief.voxel / 2)
zs = np.arange(bounds[2] - pad, bounds[5] + pad, brief.voxel / 2)
grid = (xs[:, None, None], np.full((1, 1, 1), y), zs[None, None, :])
solid = design.roof.get_distance_numpy(*grid)[:, 0, :]
kept = keep.get_distance_numpy(*grid)[:, 0, :]
fig, ax = plt.subplots(figsize=(10, 4))
ax.contourf(xs, zs, solid.T < 0, levels=[-0.5, 0.5, 1.5], colors=["white", "0.25"])
ax.contour(xs, zs, kept.T, levels=[0], colors="tab:orange")
ax.set(aspect="equal", xlabel="x (mm)", ylabel="z (mm)",
       title=f"{SOURCE}: design + keep (orange), y={y:g} mm, voxel={brief.voxel:g} mm")
plt.show()


In [ ]:
if SOURCE == "body_shell":
    summary = body_keep(result)
    assert summary["kept_samples"] > 0, "keep band unresolved at this voxel"
    assert summary["missing_samples"] == summary["keep_outside_shell_samples"] == 0
    print(json.dumps(summary, indent=2))
    print("Composition check only; physical backplate thickness/margin and T=300 unverified.")
else:
    print(result.incision.summary())
    print(roof.clear_apertures(brief, drawing, design.roof))


`scripts/m2_design.py 100 <new-run-id>` records the matching body preview in
`runs.csv`, with a diagnostic cut that deliberately removes material and then
restores keep through design. It writes results with voxel and actual spacing.
This run is on the existing test box. It does not validate the thin SubD shell,
physical backplate margin, final T/b_min, water, openings or pores (§8 unchanged).
